# CN vs LMCI vs AD MRI Classification with 3-View 2D CNN

该版本使用 `3-view 2D CNN` 方案，针对已经完成脑区提取的 MRI NIfTI 数据进行分类：
- 统一重采样到 `1 x 1 x 1 mm`
- 使用 `non-zero bounding box` 做 tight crop，并增加 margin
- 采用 `robust clipping + non-zero normalization`
- 从 `axial / coronal / sagittal` 三个方向提取中心 slab 视图
- 使用三分支 2D CNN 融合完成分类


In [1]:
import os
import random
import logging
from pathlib import Path
from datetime import datetime

os.environ["TF_ENABLE_XLA"] = "0"
os.environ["TF_XLA_FLAGS"] = "--tf_xla_auto_jit=0"
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "1")

import numpy as np
import pandas as pd
import nibabel as nib
from nibabel.processing import resample_to_output
from scipy.ndimage import zoom
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

tf.config.optimizer.set_jit(False)
print('TensorFlow:', tf.__version__)
print('GPU devices:', tf.config.list_physical_devices('GPU'))


2026-04-16 14:00:02.646500: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-04-16 14:00:02.646533: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-04-16 14:00:02.647748: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-16 14:00:03.190720: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


TensorFlow: 2.15.0
GPU devices: []


2026-04-16 14:00:03.961404: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2256] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


In [2]:
# ----------------------------
# Config
# ----------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

CN_DIR = Path('/home/biolab_374/Downloads/image_data/nii_gz/CN')
LMCI_DIR = Path('/home/biolab_374/Downloads/image_data/nii_gz/LMCI')
AD_DIR = Path('/home/biolab_374/Downloads/image_data/nii_gz/AD')

CLASS_NAMES = ['CN', 'LMCI', 'AD']
CLASS_TO_LABEL = {'CN': 0, 'LMCI': 1, 'AD': 2}
CLASS_LIMITS = {'CN': 1000, 'LMCI': 800, 'AD': 1000}

TARGET_SPACING = (1.0, 1.0, 1.0)
VIEW_SHAPE = (192, 192)
AXIAL_SLAB_THICKNESS = 5
CORONAL_SLAB_THICKNESS = 5
SAGITTAL_SLAB_THICKNESS = 5
CROP_MARGIN = 8
INTENSITY_PERCENTILES = (1, 99)

BATCH_SIZE = 16
EPOCHS = 25
LEARNING_RATE = 1e-3

RUN_TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')
LOG_DIR = Path.cwd() / 'logs'
LOG_DIR.mkdir(parents=True, exist_ok=True)
LOG_FILE = LOG_DIR / f'{RUN_TIMESTAMP}_3view2dcnn.log'
MODEL_FILE = str(Path.cwd() / 'mri_3view_2dcnn_best.keras')

logger = logging.getLogger('alz_mri_3view')
logger.setLevel(logging.INFO)
logger.handlers.clear()
file_handler = logging.FileHandler(LOG_FILE, encoding='utf-8')
file_handler.setFormatter(logging.Formatter('%(asctime)s | %(levelname)s | %(message)s'))
logger.addHandler(file_handler)
logger.propagate = False

print('Log file:', LOG_FILE)


Log file: /home/biolab_374/PycharmProjects/alz_image_classification/BIOCB26/logs/20260416_140003_3view2dcnn.log


In [3]:
def list_nifti_files(folder: Path) -> list[Path]:
    return sorted([*folder.rglob('*.nii'), *folder.rglob('*.nii.gz')])


def load_nifti_image(path: str) -> nib.Nifti1Image:
    img = nib.load(path)
    img = nib.as_closest_canonical(img)
    data = img.get_fdata(dtype=np.float32)

    if data.ndim == 4:
        data = data[..., 0]
    if data.ndim != 3:
        raise ValueError(f'Unsupported volume shape: {data.shape} @ {path}')

    return nib.Nifti1Image(data, img.affine, img.header)


def resample_image_to_1mm(img: nib.Nifti1Image) -> np.ndarray:
    resampled = resample_to_output(img, voxel_sizes=TARGET_SPACING, order=1)
    vol = resampled.get_fdata(dtype=np.float32)
    if vol.ndim != 3:
        raise ValueError(f'Resampled volume must be 3D, got: {vol.shape}')
    return vol


def nonzero_bbox(volume: np.ndarray) -> tuple[slice, slice, slice]:
    coords = np.argwhere(volume > 0)
    if coords.size == 0:
        raise ValueError('Volume contains no non-zero voxels after resampling.')

    mins = coords.min(axis=0)
    maxs = coords.max(axis=0) + 1
    starts = np.maximum(mins - CROP_MARGIN, 0)
    stops = np.minimum(maxs + CROP_MARGIN, volume.shape)
    return tuple(slice(int(start), int(stop)) for start, stop in zip(starts, stops))


def clip_and_normalize(volume: np.ndarray) -> np.ndarray:
    mask = volume > 0
    if not np.any(mask):
        raise ValueError('Tight crop contains no non-zero voxels.')

    voxels = volume[mask]
    lo, hi = np.percentile(voxels, INTENSITY_PERCENTILES)
    volume = np.clip(volume, lo, hi)
    volume = (volume - lo) / (hi - lo + 1e-8)
    volume[~mask] = 0.0
    return volume.astype(np.float32)


def central_slab_mean(volume: np.ndarray, axis: int) -> np.ndarray:

    center = volume.shape[axis] // 2

    # axis == 0:Sagittal
    if axis == 0:
        slab = volume[center-20:center+20, :, :]
    # axis == 1:Coronal
    elif axis == 1:
        slab = volume[:, center-20:center+20, :]
    # axis == 2:Axial
    else:
        slab = volume[:, :, center+20:center+60]

    return slab.mean(axis=axis).astype(np.float32)

def central_slab_mean_1(volume: np.ndarray, axis: int,
                      axial_thickness: int = AXIAL_SLAB_THICKNESS,
                      coronal_thickness: int = CORONAL_SLAB_THICKNESS,
                      sagittal_thickness: int = SAGITTAL_SLAB_THICKNESS) -> np.ndarray:

    center = volume.shape[axis] // 2
    # half = thickness // 2
    # start = max(center - half, 0)
    # stop = min(center + half + 1, volume.shape[axis])

    # axis == 0:Sagittal
    if axis == 0:
        slab = volume[center-20:center+20, :, :]
    # axis == 1:Coronal
    elif axis == 1:
        slab = volume[:, center-20:center+20, :]
    # axis == 2:Axial
    else:
        slab = volume[:, :, center+20:center+60]

    return slab.mean(axis=axis).astype(np.float32)


def resize_with_pad(image_2d: np.ndarray, target_shape: tuple[int, int] = VIEW_SHAPE) -> np.ndarray:
    src_h, src_w = image_2d.shape
    dst_h, dst_w = target_shape
    scale = min(dst_h / src_h, dst_w / src_w)
    resized = zoom(image_2d, zoom=(scale, scale), order=1)

    out = np.zeros(target_shape, dtype=np.float32)
    new_h, new_w = resized.shape
    offset_h = (dst_h - new_h) // 2
    offset_w = (dst_w - new_w) // 2
    out[offset_h:offset_h + new_h, offset_w:offset_w + new_w] = resized
    return out


def preprocess_volume(path: str) -> np.ndarray:
    img = load_nifti_image(path)
    volume = resample_image_to_1mm(img)
    bbox = nonzero_bbox(volume)
    volume = volume[bbox]
    volume = clip_and_normalize(volume)
    return volume


def volume_to_three_views(path: str) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    volume = preprocess_volume(path)

    axial = resize_with_pad(central_slab_mean(volume, axis=2))
    coronal = resize_with_pad(central_slab_mean(volume, axis=1))
    sagittal = resize_with_pad(central_slab_mean(volume, axis=0))

    return (
        axial[..., np.newaxis],
        coronal[..., np.newaxis],
        sagittal[..., np.newaxis],
    )


In [ ]:
def build_dataframe() -> pd.DataFrame:
    for folder in (CN_DIR, LMCI_DIR, AD_DIR):
        assert folder.exists(), f'Directory not found: {folder}'

    cn_files = list_nifti_files(CN_DIR)[:CLASS_LIMITS['CN']]
    lmci_files = list_nifti_files(LMCI_DIR)[:CLASS_LIMITS['LMCI']]
    ad_files = list_nifti_files(AD_DIR)[:CLASS_LIMITS['AD']]

    print(f'CN files: {len(cn_files)}')
    print(f'LMCI files: {len(lmci_files)}')
    print(f'AD files: {len(ad_files)}')

    records = []
    for p in cn_files:
        records.append({'filepath': str(p), 'label': CLASS_TO_LABEL['CN'], 'group': 'CN'})
    for p in lmci_files:
        records.append({'filepath': str(p), 'label': CLASS_TO_LABEL['LMCI'], 'group': 'LMCI'})
    for p in ad_files:
        records.append({'filepath': str(p), 'label': CLASS_TO_LABEL['AD'], 'group': 'AD'})

    df = pd.DataFrame(records)
    print(df['group'].value_counts())
    return df


def build_view_arrays(input_df: pd.DataFrame, split_name: str):
    axial_views = []
    coronal_views = []
    sagittal_views = []
    labels = []
    kept_rows = []

    for row in input_df.itertuples(index=False):
        try:
            axial, coronal, sagittal = volume_to_three_views(row.filepath)
            axial_views.append(axial)
            coronal_views.append(coronal)
            sagittal_views.append(sagittal)
            labels.append(int(row.label))
            kept_rows.append({'filepath': row.filepath, 'label': int(row.label), 'group': row.group})
        except Exception as exc:
            logger.error(
                'Failed preprocessing | split=%s | group=%s | label=%s | path=%s | error=%s',
                split_name,
                row.group,
                row.label,
                row.filepath,
                exc,
            )

    if not labels:
        raise RuntimeError(f'No valid samples available for split: {split_name}')

    arrays = {
        'axial': np.stack(axial_views).astype(np.float32),
        'coronal': np.stack(coronal_views).astype(np.float32),
        'sagittal': np.stack(sagittal_views).astype(np.float32),
    }
    labels_array = np.asarray(labels, dtype=np.int32)
    kept_df = pd.DataFrame(kept_rows)

    print(
        f"{split_name}: kept {len(labels_array)} samples | "
        f"axial={arrays['axial'].shape} coronal={arrays['coronal'].shape} sagittal={arrays['sagittal'].shape}"
    )
    return arrays, labels_array, kept_df


df = build_dataframe()

train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=SEED, stratify=df['label']
)
train_df, val_df = train_test_split(
    train_df, test_size=0.2, random_state=SEED, stratify=train_df['label']
)

print('Train:', len(train_df), train_df['label'].value_counts().to_dict())
print('Val  :', len(val_df), val_df['label'].value_counts().to_dict())
print('Test :', len(test_df), test_df['label'].value_counts().to_dict())

train_arrays, y_train, train_kept_df = build_view_arrays(train_df, 'train')
val_arrays, y_val, val_kept_df = build_view_arrays(val_df, 'val')
test_arrays, y_test, test_kept_df = build_view_arrays(test_df, 'test')

print('Kept train labels:', train_kept_df['label'].value_counts().to_dict())
print('Kept val labels  :', val_kept_df['label'].value_counts().to_dict())
print('Kept test labels :', test_kept_df['label'].value_counts().to_dict())


CN files: 1000
LMCI files: 800
AD files: 1000
group
CN      1000
AD      1000
LMCI     800
Name: count, dtype: int64
Train: 1792 {2: 640, 0: 640, 1: 512}
Val  : 448 {2: 160, 0: 160, 1: 128}
Test : 560 {0: 200, 2: 200, 1: 160}


In [ ]:
# ----------------------------
# 可视化样本（中间切片）
# ----------------------------
sample_row = None
vol = None

for row in train_kept_df.itertuples(index=False):
    try:
        vol = preprocess_volume(row.filepath)
        sample_label = int(row.label)
        sample_path = row.filepath
        sample_row = row
        break
    except Exception as exc:
        logger.error('Visualization sample failed | path=%s | error=%s', row.filepath, exc)

if vol is None:
    raise RuntimeError(f'No valid training volume could be preprocessed. Check log: {LOG_FILE}')

mid_d = vol.shape[0] // 2
mid_h = vol.shape[1] // 2
mid_w = vol.shape[2] // 2

fig, ax = plt.subplots(1, 3, figsize=(12, 4))
ax[0].imshow(vol[mid_d, :, :], cmap='gray')
ax[0].set_title('Axial')
ax[1].imshow(vol[:, mid_h, :], cmap='gray')
ax[1].set_title('Coronal')
ax[2].imshow(vol[:, :, mid_w], cmap='gray')
ax[2].set_title('Sagittal')
for a in ax:
    a.axis('off')
plt.suptitle(f'Label={int(sample_label)} | {Path(sample_path).name}')
plt.tight_layout()
plt.show()


In [ ]:
def conv2d_branch(inputs, prefix: str):
    x = keras.layers.Conv2D(32, 3, padding='same', use_bias=False, name=f'{prefix}_conv1')(inputs)
    x = keras.layers.BatchNormalization(name=f'{prefix}_bn1')(x)
    x = keras.layers.Activation('relu', name=f'{prefix}_relu1')(x)
    x = keras.layers.MaxPool2D(pool_size=2, name=f'{prefix}_pool1')(x)

    x = keras.layers.Conv2D(64, 3, padding='same', use_bias=False, name=f'{prefix}_conv2')(x)
    x = keras.layers.BatchNormalization(name=f'{prefix}_bn2')(x)
    x = keras.layers.Activation('relu', name=f'{prefix}_relu2')(x)
    x = keras.layers.MaxPool2D(pool_size=2, name=f'{prefix}_pool2')(x)
    x = keras.layers.Dropout(0.1, name=f'{prefix}_drop1')(x)

    x = keras.layers.Conv2D(128, 3, padding='same', use_bias=False, name=f'{prefix}_conv3')(x)
    x = keras.layers.BatchNormalization(name=f'{prefix}_bn3')(x)
    x = keras.layers.Activation('relu', name=f'{prefix}_relu3')(x)
    x = keras.layers.MaxPool2D(pool_size=2, name=f'{prefix}_pool3')(x)

    x = keras.layers.Conv2D(192, 3, padding='same', use_bias=False, name=f'{prefix}_conv4')(x)
    x = keras.layers.BatchNormalization(name=f'{prefix}_bn4')(x)
    x = keras.layers.Activation('relu', name=f'{prefix}_relu4')(x)
    x = keras.layers.GlobalAveragePooling2D(name=f'{prefix}_gap')(x)
    return x


def build_3view_2d_cnn(input_shape: tuple[int, int, int] = VIEW_SHAPE + (1,)) -> keras.Model:
    axial_input = keras.Input(shape=input_shape, name='axial')
    coronal_input = keras.Input(shape=input_shape, name='coronal')
    sagittal_input = keras.Input(shape=input_shape, name='sagittal')

    axial_feat = conv2d_branch(axial_input, 'axial')
    coronal_feat = conv2d_branch(coronal_input, 'coronal')
    sagittal_feat = conv2d_branch(sagittal_input, 'sagittal')

    x = keras.layers.Concatenate(name='concat_views')([axial_feat, coronal_feat, sagittal_feat])
    x = keras.layers.Dense(256, activation='relu', name='fusion_dense')(x)
    x = keras.layers.Dropout(0.3, name='fusion_dropout')(x)
    outputs = keras.layers.Dense(len(CLASS_NAMES), activation='softmax', name='predictions')(x)

    return keras.Model(
        inputs={'axial': axial_input, 'coronal': coronal_input, 'sagittal': sagittal_input},
        outputs=outputs,
        name='alz_mri_3view_2dcnn',
    )


def to_model_inputs(arrays: dict[str, np.ndarray]) -> dict[str, np.ndarray]:
    return {
        'axial': arrays['axial'],
        'coronal': arrays['coronal'],
        'sagittal': arrays['sagittal'],
    }


classes = np.sort(train_kept_df['label'].unique())
class_weights_arr = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=train_kept_df['label'].values,
)
class_weight = {int(cls): float(weight) for cls, weight in zip(classes, class_weights_arr)}
print('class_weight:', class_weight)

model = build_3view_2d_cnn()
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
    jit_compile=False,
)
model.summary()


In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_accuracy', mode='max', patience=6, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6),
    keras.callbacks.ModelCheckpoint(MODEL_FILE, monitor='val_accuracy', mode='max', save_best_only=True),
]

history = model.fit(
    to_model_inputs(train_arrays),
    y_train,
    validation_data=(to_model_inputs(val_arrays), y_val),
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    class_weight=class_weight,
    callbacks=callbacks,
    verbose=1,
)

print('History keys:', history.history.keys())


In [ ]:
test_metrics = model.evaluate(to_model_inputs(test_arrays), y_test, verbose=1)
print('Test metrics:', dict(zip(model.metrics_names, test_metrics)))

y_prob = model.predict(to_model_inputs(test_arrays), verbose=1)
y_pred = np.argmax(y_prob, axis=1)

print(classification_report(y_test, y_pred, target_names=CLASS_NAMES, digits=4))
print('Confusion Matrix:', confusion_matrix(y_test, y_pred))

y_test_onehot = keras.utils.to_categorical(y_test, num_classes=len(CLASS_NAMES))
macro_auc = roc_auc_score(y_test_onehot, y_prob, average='macro', multi_class='ovr')
weighted_auc = roc_auc_score(y_test_onehot, y_prob, average='weighted', multi_class='ovr')
print(f'Macro ROC-AUC (OvR): {macro_auc:.4f}')
print(f'Weighted ROC-AUC (OvR): {weighted_auc:.4f}')
